In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np
import copy
import random

2025-12-03 17:44:37.541969: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-03 17:44:37.542010: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-03 17:44:37.560146: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-03 17:44:37.611857: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-03 17:44:38.705124: W tensorflow/comp

In [2]:
batch_size = 128
learning_rate = 0.001

In [3]:
@tf.keras.saving.register_keras_serializable()
class MLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense2 = tf.keras.layers.Dense(units=1024, activation=tf.nn.leaky_relu)
        self.dense3 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense4 = tf.keras.layers.Dense(units=256, activation=tf.nn.leaky_relu)
        self.dense5 = tf.keras.layers.Dense(units=8)

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        x = self.dense3(x)
        x = self.dense4(x)
        output = self.dense5(x)
        return output

In [4]:
class ParaServer:
    def __init__(self):
        self.model = MLP()
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    def upload(self, grads):
        self.optimizer.apply_gradients(grads_and_vars=zip(grads, self.model.variables))
        return self.model
    def download(self):
        return self.model
    def initModel(self, x):
        self.model(x)

In [5]:
def valiAll(index_epoch):
    m = ps.download()
    model = copy.deepcopy(m)
    y_v_p = model(X_v)
    va_mse = tf.reduce_mean(tf.square(y_v_p - y_v))
    va_rmse = tf.sqrt(va_mse)
    va_mae = tf.reduce_mean(tf.abs(y_v_p - y_v))
    va_r2 = 1 - tf.reduce_sum(tf.square(y_v_p - y_v)) / tf.reduce_sum(tf.square(y_v - tf.reduce_mean(y_v)))
    print("mse:{} rmse:{} mae:{} r2:{}".format(va_mse, va_rmse, va_mae, va_r2))
    r2sv[index_epoch] = va_r2.numpy()

In [6]:
class Node:
    def __init__(self, dsName, freq, mu=1e-4):
        self.freq = freq
        self.model = MLP()
        self.mu = mu
        self.dataset = pd.read_csv(dsName, encoding='utf-8').sample(frac=1).reset_index(drop=True)
        self.X = self.dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
        self.y = self.dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)
        self.dataset_train = tf.data.Dataset.from_tensor_slices((self.X, self.y))
        self.dataset_train = self.dataset_train.shuffle(buffer_size=23000)
        self.dataset_train = self.dataset_train.batch(batch_size)
        self.dataset_train = self.dataset_train.prefetch(tf.data.experimental.AUTOTUNE)
    def train(self, index_epoch):
        m = ps.download()
        self.model = copy.deepcopy(m)
        global_weights = [tf.identity(w) for w in self.model.trainable_variables]
        for X, y in self.dataset_train:
            with tf.GradientTape() as tape:
                y_pred = self.model(X)
                tr_mse = tf.reduce_mean(tf.square(y_pred - y))
                prox_term = tf.add_n([
                        tf.nn.l2_loss(w - w0)
                        for w, w0 in zip(self.model.trainable_variables, global_weights)
                    ])
                loss = tr_mse + self.mu * prox_term
            tr_rmse = tf.sqrt(tr_mse)
            tr_mae = tf.reduce_mean(tf.abs(y_pred - y))
            tr_r2 = 1 - tf.reduce_sum(tf.square(y_pred - y)) / tf.reduce_sum(tf.square(y - tf.reduce_mean(y)))
            grads = tape.gradient(loss, self.model.variables)
            m = ps.upload(grads)
            self.model = copy.deepcopy(m)
        # if epoch_index in np.arange(0, num_epochs, 25).tolist() or epoch_index == num_epochs - 1:
        if True:
            print("node:{} epoch:{}".format(self.freq, index_epoch))
            print("train mse:{} rmse:{} mae:{} r2:{}".format(tr_mse, tr_rmse, tr_mae, tr_r2))
            r2s[self.freq][index_epoch] = tr_r2.numpy()

In [7]:
r2s = {2.4:{},2.5:{},2.6:{}}
r2sv = {}

In [8]:
test_dataset = pd.read_csv("Test.csv", encoding='utf-8').sample(frac=1).reset_index(drop=True)
X_v = test_dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
y_v = test_dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)

In [9]:
ps = ParaServer()
ps.initModel(X_v)

2025-12-03 12:37:01.856814: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 647 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:17:00.0, compute capability: 7.5
2025-12-03 12:37:01.858046: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 9400 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:65:00.0, compute capability: 7.5


In [10]:
nodeList = [Node('./24Train.csv', 2.4), Node('./25Train.csv', 2.5), Node('./26Train.csv', 2.6)]

In [ ]:
orders = [0, 1, 2]
turn = [np.array([[26, 104], [178, 312], [344, 464], [520, 600]]), np.array([[0, 94], [149, 223], [319, 433], [464, 580]]), np.array([[32, 151], [155, 248], [270, 354], [378, 502]])]
for i in range(600):
    random.shuffle(orders)
    for j in orders:
        for l, r in turn[j]:
            if l <= i < r:
                nodeList[j].train(i)
    valiAll(i)

2025-12-03 12:37:02.771809: I external/local_xla/xla/service/service.cc:168] XLA service 0x5babd5b065a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-12-03 12:37:02.771833: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5
2025-12-03 12:37:02.771840: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (1): NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5
2025-12-03 12:37:02.777417: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-12-03 12:37:02.795855: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8902
I0000 00:00:1764765422.879260   24168 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


node:2.5 epoch:0
train mse:0.098273865878582 rmse:0.31348663568496704 mae:0.2565968930721283 r2:0.1764456033706665
mse:0.09891654551029205 rmse:0.3145100176334381 mae:0.2535741627216339 r2:0.18118047714233398
node:2.5 epoch:1
train mse:0.08322405815124512 rmse:0.2884857952594757 mae:0.23255270719528198 r2:0.3094068169593811
mse:0.0845930203795433 rmse:0.2908487915992737 mae:0.2378758192062378 r2:0.299748957157135
node:2.5 epoch:2
train mse:0.0739520713686943 rmse:0.27194130420684814 mae:0.22179026901721954 r2:0.38551217317581177
mse:0.08074042201042175 rmse:0.28414860367774963 mae:0.2276175171136856 r2:0.3316403031349182
node:2.5 epoch:3
train mse:0.08787383139133453 rmse:0.2964352071285248 mae:0.2384551614522934 r2:0.2658194899559021
mse:0.08783219754695892 rmse:0.296364963054657 mae:0.23819835484027863 r2:0.2729353904724121
node:2.5 epoch:4
train mse:0.07133941352367401 rmse:0.26709437370300293 mae:0.2171788215637207 r2:0.41036146879196167
mse:0.07190587371587753 rmse:0.2681527137756

In [ ]:
for i in r2sv:
    print(i)

0.5486207
0.5958206
0.6077136
0.6231611
0.6104752
0.6293508
0.6425981
0.6651129
0.6868254
0.70473087
0.7077899
0.7163201
0.74149126
0.7564358
0.7607001
0.7586043
0.7359556
0.77295303
0.7995131
0.79581016
0.81139106
0.79567385
0.7558851
0.7826637
0.7788899
0.7793881
0.8076835
0.8089497
0.8327596
0.81962246
0.7543961
0.7840984
0.81794167
0.79270005
0.82521594
0.78134686
0.83092093
0.80101323
0.8339965
0.7948794
0.84356713
0.8465845
0.79730135
0.83424723
0.8134524
0.80595034
0.83938706
0.846708
0.80759996
0.84521496
0.85964924
0.81562316
0.7884283
0.8559452
0.82728237
0.8672112
0.859355
0.8188611
0.8648717
0.86271995
0.79972637
0.85287476
0.8543722
0.8266123
0.82080746
0.8168412
0.8181273
0.8719065
0.82265854
0.8701856
0.8311549
0.8286662
0.8626292
0.87516916
0.8826012
0.8897865
0.8933047
0.89149964
0.89433753
0.89729017
0.8999928
0.8984779
0.9020348
0.88937795
0.90497017
0.8998585
0.89999324
0.9050486
0.90359885
0.9058355
0.91599554
0.9061472
0.9080994
0.90987
0.9071824
0.9126997
0.91493